# 01. Data, visualisation and split
Downloads the Makerere cassava photos, shows the class distribution and sample images, detects near-duplicates, and writes the published 70/15/15 split. Run in Google Colab with a T4 runtime; upload `kaggle.json` first.

In [ ]:
# Environment
!pip install --quiet kaggle imagehash scikit-learn
import os, shutil, json
os.makedirs("/root/.kaggle", exist_ok=True)
shutil.copy("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)
print("kaggle token in place")

In [ ]:
# Download the competition data (accept the rules on the Kaggle site first, or this is refused)
!kaggle competitions download --competition cassava-leaf-disease-classification --path /content/data
!unzip -q -o /content/data/cassava-leaf-disease-classification.zip -d /content/data
import pandas as pd
labels = pd.read_csv("/content/data/train.csv")
label_names = json.load(open("/content/data/label_num_to_disease_map.json"))
SHORT = {0: "CBB", 1: "CBSD", 2: "CGM", 3: "CMD", 4: "healthy"}
labels["class"] = labels["label"].map(SHORT)
print(len(labels), "images"); print(labels["class"].value_counts())

## Class distribution
CMD dominates, which is why training uses class weights and why macro-F1, not accuracy, is the headline metric.

In [ ]:
import matplotlib.pyplot as plt
counts = labels["class"].value_counts().reindex(["CBB", "CBSD", "CGM", "CMD", "healthy"])
plt.figure(figsize=(6, 3.2)); plt.bar(counts.index, counts.values, color="#2F4F6F")
for i, v in enumerate(counts.values): plt.text(i, v + 150, f"{v:,}", ha="center", fontsize=9)
plt.title("Makerere cassava dataset: images per class"); plt.ylabel("images"); plt.tight_layout(); plt.show()
print((counts / counts.sum() * 100).round(1).astype(str) + " %")

## Sample images per class
Field photos: varied lighting, backgrounds and leaf ages. The label names in the competition file call class 2 'Green Mottle'; the dataset authors (Mwebaze et al., 2019) call it green mite, and this project follows the authors.

In [ ]:
from PIL import Image
image_directory = "/content/data/train_images"
fig, axes = plt.subplots(5, 4, figsize=(9, 11))
for row, name in enumerate(["CBB", "CBSD", "CGM", "CMD", "healthy"]):
    for col, image_id in enumerate(labels[labels["class"] == name].sample(4, random_state=42)["image_id"]):
        axes[row, col].imshow(Image.open(f"{image_directory}/{image_id}")); axes[row, col].axis("off")
        if col == 0: axes[row, col].set_title(name, loc="left", fontsize=10)
plt.tight_layout(); plt.show()

## Image sizes
Checks that one resize rule fits the whole set.

In [ ]:
import random
sample_ids = random.Random(42).sample(list(labels["image_id"]), 500)
sizes = pd.DataFrame([Image.open(f"{image_directory}/{i}").size for i in sample_ids], columns=["width", "height"])
print(sizes.describe().loc[["min", "50%", "max"]])

## Near-duplicate detection
A perceptual hash gives near-identical photos the same fingerprint, so copies stay together and never cross the train/validation/test boundary.

In [ ]:
import imagehash
from tqdm import tqdm
labels["phash"] = [str(imagehash.phash(Image.open(f"{image_directory}/{i}").convert("RGB"), hash_size=8)) for i in tqdm(labels["image_id"])]
group_size = labels.groupby("phash")["image_id"].transform("count")
print("images sharing a fingerprint with another image:", int((group_size > 1).sum()))
print("distinct fingerprints:", labels["phash"].nunique())

## Stratified group split, 70 / 15 / 15, seed 42
Stratified by class so that bacterial blight is in every split; grouped by fingerprint so duplicates stay together.

In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold
labels["split"] = "train"
folds = list(StratifiedGroupKFold(n_splits=20, shuffle=True, random_state=42).split(labels, labels["label"], groups=labels["phash"]))
labels.loc[np.concatenate([folds[i][1] for i in range(0, 3)]), "split"] = "validation"
labels.loc[np.concatenate([folds[i][1] for i in range(3, 6)]), "split"] = "test"
split_table = labels.groupby(["split", "class"]).size().unstack().reindex(["train", "validation", "test"])
print(split_table); print(split_table.sum(axis=1))
labels[["image_id", "label", "class", "phash", "split"]].to_csv("split_v1_seed42.csv", index=False)
print("saved split_v1_seed42.csv; commit it to data/ in the repository")